In [1]:
# Imports and special token IDs used by both tokenizers and the model.
from datasets import load_dataset
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PAD, BOS, EOS, UNK = range(4)
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]


In [2]:
# Stream small subsets so the full WMT14 training set is not downloaded.
# Increase TRAIN_COUNT gradually if time and memory allow.
TRAIN_COUNT = 50_000
VALIDATION_COUNT = 200
TEST_COUNT = 200

train_stream = load_dataset("wmt/wmt14", "fr-en", split="train", streaming=True)
validation_stream = load_dataset("wmt/wmt14", "fr-en", split="validation", streaming=True)
test_stream = load_dataset("wmt/wmt14", "fr-en", split="test", streaming=True)

train_sample = list(train_stream.take(TRAIN_COUNT))
validation_sample = list(validation_stream.take(VALIDATION_COUNT))
test_sample = list(test_stream.take(TEST_COUNT))
print(f"Training: {len(train_sample)}, validation: {len(validation_sample)}, test: {len(test_sample)}")


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Training: 50000, validation: 200, test: 200


In [3]:
# The dataset stores each pair as French and English. This notebook translates
# French input into English output, so train one tokenizer for each direction.
def train_bpe(texts, vocab_size=8_000):
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL_TOKENS)
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

fr_tokenizer = train_bpe(row["translation"]["fr"] for row in train_sample)
en_tokenizer = train_bpe(row["translation"]["en"] for row in train_sample)

def encode(text, tokenizer, max_len=40):
    # Add sentence boundary IDs and cap the sequence length.
    token_ids = tokenizer.encode(text).ids[:max_len - 2]
    return [BOS] + token_ids + [EOS]


In [4]:
# Make padded French source and English target tensors for each mini-batch.
def collate_batch(rows):
    pairs = [
        (encode(row["translation"]["fr"], fr_tokenizer),
         encode(row["translation"]["en"], en_tokenizer))
        for row in rows
    ]
    src_len = max(len(src) for src, _ in pairs)
    tgt_len = max(len(tgt) for _, tgt in pairs)
    src = torch.full((len(rows), src_len), PAD, dtype=torch.long)
    tgt = torch.full((len(rows), tgt_len), PAD, dtype=torch.long)
    for i, (src_ids, tgt_ids) in enumerate(pairs):
        src[i, :len(src_ids)] = torch.tensor(src_ids)
        tgt[i, :len(tgt_ids)] = torch.tensor(tgt_ids)
    return src, tgt

train_loader = DataLoader(train_sample, batch_size=32, shuffle=True, collate_fn=collate_batch)


In [5]:
# Sinusoidal position vectors give the Transformer information about token order.
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-torch.log(torch.tensor(10_000.0)) / d_model)
        )
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(1))

    def forward(self, x):
        return x + self.pe[:x.size(0)]


In [6]:
# Encoder-decoder model: French token IDs in, English token scores out.
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=384, nhead=12,
                 num_layers=4, dim_feedforward=1536, dropout=0.2):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD)
        self.positional_encoding = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_layers, num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=False,
        )
        self.output = nn.Linear(d_model, tgt_vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt_input):
        # 2D padding masks are shaped (batch, sequence length).
        src_padding_mask = src.eq(PAD)
        tgt_padding_mask = tgt_input.eq(PAD)
        src = self.src_embedding(src).transpose(0, 1) * self.d_model ** 0.5
        tgt = self.tgt_embedding(tgt_input).transpose(0, 1) * self.d_model ** 0.5
        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)
        # Causal mask prevents the decoder from looking at future French tokens.
        causal_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(0), device=tgt.device)
        result = self.transformer(
            src, tgt, tgt_mask=causal_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )
        return self.output(result).transpose(0, 1)


In [7]:
# Create a new model because its embedding indices must match the BPE tokenizers.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Seq2SeqTransformer(
    fr_tokenizer.get_vocab_size(),  # French is the source language.
    en_tokenizer.get_vocab_size(),  # English is the target language.
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
print("Training device:", device)


Training device: cuda


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


In [8]:
# Validation next-token accuracy: teacher-forced token score, ignoring padding.
def evaluate_next_token_accuracy(rows, batch_size=32):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            src, tgt = collate_batch(rows[start:start + batch_size])
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src, tgt[:, :-1])
            expected = tgt[:, 1:]
            mask = expected.ne(PAD)
            correct += ((logits.argmax(dim=-1) == expected) & mask).sum().item()
            total += mask.sum().item()
    return correct / total if total else 0.0


In [9]:
# Train and check validation accuracy at the end of every epoch.
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_input, tgt_expected = tgt[:, :-1], tgt[:, 1:]
        optimizer.zero_grad()
        logits = model(src, tgt_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_expected.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    val_accuracy = evaluate_next_token_accuracy(validation_sample)
    print(f"Epoch {epoch + 1}: loss={total_loss / len(train_loader):.3f}, validation next-token accuracy={val_accuracy:.2%}")


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 1: loss=4.076, validation next-token accuracy=27.66%


In [10]:
# Greedy decoding: generate an English subword at each step until EOS.
def translate_to_english(sentence, max_len=40):
    model.eval()
    src_ids = encode(sentence, fr_tokenizer)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    generated = [BOS]
    with torch.no_grad():
        for _ in range(max_len):
            tgt = torch.tensor([generated], dtype=torch.long, device=device)
            next_id = model(src, tgt)[0, -1].argmax().item()
            if next_id == EOS:
                break
            generated.append(next_id)
    return en_tokenizer.decode(generated[1:])

row = validation_sample[0]
print("French:", row["translation"]["fr"])
print("Reference English:", row["translation"]["en"])
print("Model English:", translate_to_english(row["translation"]["fr"]))


French: Une stratégie républicaine pour contrer la réélection d'Obama
Reference English: A Republican strategy to counter the re-election of Obama
Model English: A new strategy for the new economy is to be able to make a more effective way of achieving its own resources


In [11]:
# Use the held-out test sample only for final evaluation.
test_accuracy = evaluate_next_token_accuracy(test_sample)
print(f"Test next-token accuracy: {test_accuracy:.2%}")


Test next-token accuracy: 11.63%
